In [1]:
import numpy as np
import math
from datetime import datetime, timedelta
from dash import Dash, html, dcc
from data.data_tools import Database, SENSOR_LIST, ddmm_to_decimal
from src.data.data_utils import xyz_array
from src.funcs import ekf_navigation, ekf_to_coor

app = Dash()

sample_limit = 100
db = Database("../data/dataset.db")

gps_readings = db.get_gps_readings(start_datetime=datetime(2026, 3, 20, 1), end_datetime=datetime(2026, 3, 21), min_satellite_count=4)

# subsample list to defined count starting at a zero velocity sample
for idx, reading in enumerate(gps_readings):
    if reading.speed <= 0.05 and idx + 1 < len(gps_readings) and gps_readings[idx + 1].speed <= 0.05:
        gps_readings = gps_readings[idx:idx+sample_limit]
        break

for idx, reading in enumerate(gps_readings):
    if idx > 0 and reading.timestamp - gps_readings[idx-1].timestamp > timedelta(seconds=1.1):
        gps_readings = gps_readings[0:idx]
        break

imu_readings = db.get_imu_readings(start_datetime=gps_readings[0].timestamp, end_datetime=gps_readings[-1].timestamp)

gps_coor = [(ddmm_to_decimal(gps.latitude), ddmm_to_decimal(gps.longitude)) for gps in gps_readings]
start_lat, start_lon = gps_coor[0]

ekf_pos_map = {}
ekf_vel_map = {}
ekf_coor_map = {}
min_sample = math.inf

print(imu_readings[0])

for sensor in SENSOR_LIST:

    # decompose sensor readings to separate arrays
    sensor_readings = [r for r in imu_readings if r.sensor_name == sensor]
    gyr = np.array([np.array(r.raw_gyr) for r in sensor_readings])
    acc = np.array([np.array(r.raw_acc) for r in sensor_readings])
    mag = np.array([np.array(r.raw_mag) for r in sensor_readings])
    time = np.array([r.timestamp for r in sensor_readings])

    print(gyr)
    print(f"Running EKF on sensor {sensor} - {len(sensor_readings)} readings")
    print(f"Initial GPS data:"
          f"\n\tposition: {start_lat:.2f}, {start_lon:.2f}"
          f"\n\tcourse: {gps_readings[0].course}"
          f"\n\tsamples: {len(gps_coor)}")
    print(f"Initial IMU data:"
          f"\n\tmag heading: {math.degrees(math.atan2(mag[0, 0], mag[0, 1]))}"
          f"\n\tsamples: {len(sensor_readings)}"
          f"\n\trate: {len(sensor_readings)/len(gps_coor)}")

    Q, V, P = ekf_navigation(gyr, acc, mag, time, mag_ref=[.237617,	.45396,	.383970])

    ekf_coor_map[sensor] = ekf_to_coor(start_lat, start_lon, P)
    min_sample = min(min_sample, len(ekf_coor_map[sensor]))

    ekf_pos_map[sensor] = P
    ekf_vel_map[sensor] = V

stacked = np.array([ekf_pos_map[s][:min_sample] for s in SENSOR_LIST])  # shape (6, min_sample, 3)
avg_ekf = stacked.mean(axis=0)

# ekf_coor_map["avg"] = ekf_to_coor(start_lat, start_lon, avg_ekf)

IMUReading(gps_second=10718, gps_time_elapsed=1001125, sensor_name='icm_20948_1', sensor_type='icm_20948', sensor_id=1, source_time=193235, raw_gyr=array([-0.0042569 ,  0.00638535,  0.00532113]), raw_acc=array([-0.6799533, -3.4237084,  9.260772 ]), raw_mag=array([ 68.55, -88.95,  88.35]), raw_tmp=38.587685, gyr=array([0., 0., 0.]), acc=array([0., 0., 0.]), mag=array([0., 0., 0.]), tmp=0.0, timestamp=datetime.datetime(2026, 3, 20, 1, 7, 19, 1125))
[[-0.00855211 -0.03298672 -0.00488692]
 [-0.00733038 -0.03176499 -0.00366519]
 [-0.00733038 -0.03176499 -0.00366519]
 ...
 [-0.00244346 -0.03420845 -0.00366519]
 [-0.01466076 -0.03543018 -0.00488692]
 [-0.01099557 -0.03298672 -0.00488692]]
Running EKF on sensor lsm6dsox_3 - 10364 readings
Initial GPS data:
	position: 34.17, -119.22
	course: 182.32
	samples: 100
Initial IMU data:
	mag heading: 176.49434261334432
	samples: 10364
	rate: 103.64
	EKF Checkpoint 1000
		Acceleration Estimate: [-0.01522594  0.00357562  0.00854298]
		Velocity Estimate:

In [2]:
import pandas as pd
import plotly.graph_objects as go
from dash import Dash, html, dcc
import numpy as np

# --- Assumed existing data structures ---
# gps_coor, ekf_coor_map, ekf_vel_map are provided from your logic

# 1. CREATE THE MAP FIGURE (Your existing logic)
gps_lats, gps_lons = zip(*gps_coor)
center_lat = np.mean(gps_lats)
center_lon = np.mean(gps_lons)

map_fig = go.Figure()

# Add GPS Trace
df = pd.DataFrame({
    'lat': gps_lats,
    'lon': gps_lons,
    'time': [(r.timestamp - gps_readings[0].timestamp).total_seconds() for r in gps_readings]
})
map_fig.add_trace(go.Scattermap(
    lat=df['lat'], lon=df['lon'], mode='lines+markers', name='GPS',
    line=dict(color='green', width=2), marker=dict(size=6, color='green'),
    customdata=df[['time']],
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>" +
        "Location: (%{lat}, %{lon})" +
        "<extra></extra>" # Removes the secondary box with trace name
    )
))

# Add Start Point
map_fig.add_trace(go.Scattermap(
    lat=[gps_lats[0]], lon=[gps_lons[0]], mode='markers', name='Start',
    marker=dict(size=12, color='green'),
))

# Add EKF Traces to Map
for sensor in ekf_coor_map.keys():
    ekf_coor = ekf_coor_map[sensor]

    sensor_readings = [r for r in imu_readings if r.sensor_name == sensor]
    time = [(r.timestamp - gps_readings[0].timestamp).total_seconds() for r in sensor_readings]

    df = pd.DataFrame({
        'lat': [r[0] for r in ekf_coor_map[sensor]],
        'lon': [r[1] for r in ekf_coor_map[sensor]],
        'time': time
    })

    # ekf_lats, ekf_lons, _ = zip(*ekf_coor_map[sensor])
    map_fig.add_trace(go.Scattermap(
        lat=df['lat'], lon=df['lon'], mode='lines+markers', name=f'EKF: {sensor}',
        line=dict(color='royalblue', width=2), marker=dict(size=6, color='royalblue'),customdata=df[['time']],
        hovertemplate=(
        "<b>%{customdata[0]}</b><br>" +
        "Location: (%{lat}, %{lon})" +
        "<extra></extra>" # Removes the secondary box with trace name
        )
    ))

map_fig.update_layout(
    title='EKF vs GPS Track Comparison',
    map=dict(style='open-street-map', center=dict(lat=center_lat, lon=center_lon), zoom=15),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=0, r=0, t=40, b=0),
)

# 2. CREATE THE VELOCITY LINE CHART FIGURE (The new part)
vel_fig = go.Figure()

axis_colors = {
    'x': 'royalblue',
    'y': 'crimson',
    'z': 'forestgreen'
}

for sensor, velocities in ekf_vel_map.items():
    # We use an index as the X-axis (Time/Sample count)
    # If you have a timestamp array for each sensor, replace np.arange with that.
    v_arr = np.array(velocities)
    time_axis = [imu.timestamp for imu in imu_readings if
                 gps_readings[0].timestamp <= imu.timestamp <= gps_readings[-1].timestamp and (
                             imu.sensor_name == sensor)]
    # time_axis = np.arange(len(v_arr))
    # Extract components
    vx = v_arr[:, 0]
    vy = v_arr[:, 1]
    vz = v_arr[:, 2]

    # Add Trace for X-component
    vel_fig.add_trace(go.Scatter(
        x=time_axis, y=vx,
        mode='lines',
        name=f'{sensor} (N)',
        line=dict(color=axis_colors['x'], width=1.5),
        opacity=0.8  # Slightly transparent to handle overlapping lines
    ))

    # Add Trace for Y-component
    vel_fig_trace_y = vel_fig.add_trace(go.Scatter(
        x=time_axis, y=vy,
        mode='lines',
        name=f'{sensor} (E)',
        line=dict(color=axis_colors['y'], width=1.5),
        opacity=0.8
    ))

    # Add Trace for Z-component
    vel_fig.add_trace(go.Scatter(
        x=time_axis, y=vz,
        mode='lines',
        name=f'{sensor} (D)',
        line=dict(color=axis_colors['z'], width=1.5),
        opacity=0.8
    ))



vn = [gps.speed * np.cos(np.radians(gps.course) )for gps in gps_readings]
ne = [gps.speed * np.sin(np.radians(gps.course) )for gps in gps_readings]
time_axis = [gps.timestamp for gps in gps_readings]


vel_fig.add_trace(go.Scatter(
    x=time_axis, y=vn,
    mode='lines',
    name='GPS (N)',
    line=dict(color="purple", width=1.5),
    opacity=0.8
))

vel_fig.add_trace(go.Scatter(
    x=time_axis, y=ne,
    mode='lines',
    name='GPS (E)',
    line=dict(color="goldenrod", width=1.5),
    opacity=0.8
))

vel_fig.update_layout(
    title='Sensor Velocity Over Time',
    xaxis_title='Sample Index (Time)',
    yaxis_title='Velocity (m/s)',
    legend=dict(x=0.01, y=0.99),
    template='plotly_white',
    margin=dict(l=40, r=20, t=50, b=40)
)

# 3. DASH APP LAYOUT
app = Dash(__name__)

app.layout = html.Div([
    html.H1("IMU/EKF Navigation Dashboard", style={'textAlign': 'center'}),

    # Container for the Map
    html.Div([
        dcc.Graph(figure=map_fig)
    ], style={'padding': '10px'}),

    # Container for the Velocity Chart
    html.Div([
        dcc.Graph(figure=vel_fig)
    ], style={'padding': '10px'})
])

app.run()